In [1]:
import sys
import os
from pathlib import Path
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/cs189/hw/hw2
    %pip install -r ./requirements.txt
import plotly.io as pio
# Notebook MIME output works locally without a Chrome installation.
pio.renderers.default = 'plotly_mimetype'

D:\Anaconda\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [2]:
# Initialize Otter
import otter
grader = otter.Notebook(tests_dir=None, nb_path="arena_style_control.ipynb")

D:\Anaconda\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning:

Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.



<link rel="stylesheet" href="berkeley.css">

<h1 class="cal cal-h1">Homework 02 – Welcome to the Arena (Style Control)</h1>

CS 189, Fall 2025

In this homework you will get more experience with logistic regression to create model leaderboards.

We will be taking real data from [LMArena](https://lmarena.ai/), a popular platform for crowdsourcing evaluations of large language models and recreating their leaderboards, with a few fun extra steps along the way.

The chats can be viewed interactively by accessing [ChatBot-Arena-Viewer](https://huggingface.co/spaces/BerkeleyML/Chatbot-Arena-Viewer) through hugging face. Much of the first half of this homework was first written by Prof Gonzalez back when his students first started the project, and now LMArena is a standard evaluation for large language models and turned into a company! Don't let anyone tell you logistic regression isn't valuable, it's worth at least $600 Million.
    
---


## Due Date: Friday, October 17, 11:59 PM

This assignment is due on **Friday, October 17, 11:59 PM**. You must submit your work to Gradescope by this deadline. Please refer to the syllabus for the [Slip Day policy](https://eecs189.org/fa25/syllabus/#slip-days). No late submissions will be accepted beyond the details outlined in the Slip Day policy.

### Submission Tips
- **Plan ahead**: We strongly encourage you to submit your work several hours before the deadline. This will give you ample time to address any submission issues.
- **Reach out for help early**: If you encounter difficulties, contact course staff well before the deadline. While we are happy to assist with submission issues, we cannot guarantee responses to last-minute requests.
      
<!-- --- -->

### Key Learning Objectives

In this homework you will build on the previous warmup section, implementing the Bradley-Terry ranking used in the actual Arena and taking account of style controls for model rank. In particular, you will:
1. Apply the Bradley–Terry model to build leaderboards
2. Practice analyzing conversational data and extracting stylistic features  
3. Build custom features (length, punctuation, phrase presence, etc.) and integrate them into ranking models  
4. Explore confounding stylistic variables in LLM evaluation (style vs. content)  
5. Apply pairwise evaluation methods to understand how style affects outcomes  
  
---

### Collaboration Policy
You are encouraged to discuss high-level concepts with your peers. However:
- All submitted work must be written in your own words and code.
- Do not share or copy solutions directly.
- List any collaborators (students you worked with) in the line below:

**Your Collaborators**: No human collaborators were specified in this study session.

### AI Tools Usage Disclosure
We allow the use of AI tools (e.g., ChatGPT, Copilot) **only as support**, not as a replacement for your own reasoning. To ensure transparency, you must acknowledge any use of AI tools.

Please complete one of the following:
- **A) I did not use any AI tools for this homework.**
- **B) I used AI tools in the following way(s):**  
  (describe briefly, e.g., “Used ChatGPT to get hints for debugging a NumPy indexing error”)


**Your Answer**: B) Used OpenAI Codex to discuss derivations, implement code, debug, run experiments, and draft explanations for this personal study repository.
    
---

### Grading Breakdown

| Question | Manual Grading? | Points |
|----------|-----------------|--------|
| q4a      | No              | 2      |
| q4b      | No              | 2      |
| q4c      | Yes             | 2      |
| q5a      | No              | 2      |
| q5b      | No              | 2      |
| q7a      | No              | 2      |
| q7b      | No              | 2      |
| q8a      | No              | 2      |
| q8afrq   | Yes             | 2      |
| q8b      | No              | 2      |
| q8c      | No              | 2      |
| q9a      | No              | 2      |
| q9b      | Yes             | 4      |
| q9c      | Yes             | 6      |
| **Total**|                 | **34** |

In [3]:
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotting_utils import plot_rank_heatmap, plot_style_features
#set fixed seed of 189
np.random.seed(189)

In [4]:
# ! pip install ipywidgets
# from huggingface_hub import notebook_login
# notebook_login()

In [5]:
from datasets import load_dataset
local_data = os.environ.get('HW2_ARENA_PARQUET')
if local_data:
    battles = pd.read_parquet(local_data)
else:
    ds = load_dataset('lmarena-ai/arena-human-preference-100k')
    battles = ds['train'].to_pandas()
print(f'Loaded {len(battles):,} real Arena battles.')

Loaded 106,134 real Arena battles.


In [6]:
print("Before dedup: ", len(battles))
battles = battles[battles["dedup_tag"].apply(lambda x: x.get("sampled", False))]
print("After dedup: ", len(battles))

Before dedup:  106134
After dedup:  101869


## Initialize with HW2 Warmup

Fill in the cells in this section with your implementations from part 1 of the homework.

#### HW2 Part 1: Select models and decisive battles

In [7]:
models = pd.concat([battles['model_a'],battles['model_b']]).value_counts()
selected_models = models.head(20).index.tolist()

def subselect_battles(battles, selected_models):
    mask = battles['model_a'].isin(selected_models) & battles['model_b'].isin(selected_models)
    selected_battles = battles.loc[mask].copy()
    decisive = selected_battles['winner'].isin(['model_a', 'model_b'])
    return selected_battles, selected_battles.loc[decisive].copy()

selected_battles, selected_battles_no_ties = subselect_battles(battles, selected_models)

##  **Question 4: Model Strengths**

In the earlier part of the homework, we calculated the Average Model Win-Rate.

However, this method is not ideal for our use case where battle counts per model are not equal. For instance, if ChatGPT-4o-latest battled more often with weaker models, it would have a high win rate without being an actually stronger model. Now let's explore how we can instead *learn* these model strengths.

**To recap,** we want to construct a leaderboard by assigning a strength score $S_m$ to each model $m \in \{1,...,M\}$, such that:
- The ranking reflects the probability of one model winning against another.
- For any pair of models A and B, the probability that A beats B, should depend on the *difference* in their strengths: $S_A - S_B$. Why the difference? Since we are measuring pairwise preference, there is no absolute measure of strength but rather a model's strength *relative* to other models.

**Formally, we want a function $f$ such that**
- For models A and B with scores $S_A$ and $S_B$, we want:
  $$ P(\text{A beats B}) = f(S_A - S_B) $$
- The function $f$ should be increasing (bigger skill gap, higher win chance), and always output a probability between 0 and 1.

At this point, a natural question is: what should we choose for the function $f$? A standard and effective choice is the logistic (sigmoid) function:

$$ P(\text{A beats B}) = \sigma(S_A - S_B) = \frac{1}{1 + e^{-(S_A - S_B)}} $$

Notice that this is exactly the same form as logistic regression, where the model scores are the parameters to be learned. In other words, learning model strengths from pairwise outcomes is equivalent to fitting a logistic regression model to the data.

So, we can use logistic regression to learn the model strengths that best explain the observed battle outcomes. The higher a model's score, the more likely it is to win against others. The methodology is called the [Bradley-Terry](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) model and is the underlying theory to other common scoring systems like ELO ratings.

#### Step 1: Understanding the formulation of the problem as features

To learn these model strengths, we need to prepare our data in a form suitable for logistic regression. Recall that each battle involves two models: A and B. One of them wins (for simplicity we still start by removing any battles that end in ties).

We want to convert this into:

1. A feature vector indicating which two models were involved.
2. A label representing the winner.

Each row produces two training examples:

1. One with model A as +1 and model B as –1, and a label denoting if model A wins
2. Another with model B as +1 and model A as –1, and a label denoting if model B wins

Let's take a look an example:

`row = {'model_a': 'gpt-4o-2024-05-13', 'model_b': 'claude-3-opus-20240229', 'winner': 'model_a'}`

This generates two features, which are...

**Feature 1:** [1, -1]
* +1 at index 0 (gpt-4o)
* –1 at index 1 (claude-3-opus)
* Label: 1 because model A (gpt-4o) won

**Feature 2:** [-1, 1]
* -1 at index 0 (gpt-4o)
* +1 at index 1 (claude-3-opus)
* Label: 0 because model B (claude-3-opus) lost

Why do we have to do this?
This lets the model take account for both ways:


$$ P(\text{GPT-4o beats Claude-3-opus}) = \sigma(S_{\text{GPT-4o}}  - S_{\text{Claude}})$$



$$ P(\text{Claude-3-opus beats GPT-4o}) = \sigma(S_{\text{Claude}} - S_{\text{GPT-4o}}  )$$


**Stop and Think:** When we have more than two models, how should we handle the models that were not invovled in the battle?

---

#### Step 2: Constructing generalized features and labels

Cool, now we want to generalize this formulation to the pair of not only GPT-4o and Claude-3-opus, but all the models.

Once we have turned all battle outcomes into feature vectors, we can organize them into a **feature matrix** $\mathbf{X}$ and a **label vector** $\mathbf{y}$.

We have the model strengths we want to learn:
\begin{bmatrix}
S_A \\
S_B \\
S_C
\end{bmatrix}

And we want our model to predict:

\begin{bmatrix}
\sigma(S_A - S_C) \\
\sigma(S_B - S_A) \\
\sigma(S_B - S_C)
\end{bmatrix}



As a recap...

- $\mathbf{X}$ encodes **who played whom** and in what direction.
- $\mathbf{S}$ are the model strengths we are trying to learn.
- $\mathbf{y} = \sigma(\mathbf{X} \cdot \mathbf{S})$ gives us the predicted win probabilities.

Phew, that was a long. Now let's try to actually featurize these battles and labels!

However, before analysis, let's deduplicate common prompts like "hi" and "hello" to ensure they don't overly influence the leaderboard.

In [8]:
print("Before dedup: ", len(battles))
battles = battles[battles["dedup_tag"].apply(lambda x: x.get("sampled", False))]
print("After dedup: ", len(battles))

Before dedup:  101869
After dedup:  101869


Also, let's focus on the top 20 models like we have done for 1a

In [9]:
models = battles.model_a.value_counts().index.tolist()
selected_models = models[:20]
selected_battles = battles[battles['model_a'].isin(selected_models) & battles['model_b'].isin(selected_models)]
selected_battles_no_ties = selected_battles[~selected_battles["winner"].str.contains("tie")]

In [10]:
selected_models

['claude-3-5-sonnet-20240620',
 'gpt-4o-2024-05-13',
 'gemini-1.5-pro-api-0514',
 'gemma-2-27b-it',
 'llama-3-70b-instruct',
 'gemma-2-9b-it',
 'claude-3-opus-20240229',
 'gemini-1.5-flash-api-0514',
 'gemini-1.5-pro-exp-0801',
 'llama-3.1-405b-instruct',
 'gpt-4o-mini-2024-07-18',
 'chatgpt-4o-latest',
 'gpt-4-turbo-2024-04-09',
 'deepseek-v2-api-0628',
 'claude-3-haiku-20240307',
 'llama-3-8b-instruct',
 'llama-3.1-70b-instruct',
 'llama-3.1-8b-instruct',
 'qwen2-72b-instruct',
 'deepseek-coder-v2']

## **Question 4a**

In order to train our model, we should first featurize our battles as discussed before.

**Task:** Implement the function below to transform `selected_battles_no_ties` and `selected_models` into feature vectors and labels. This will allow us to represent each battle as input-output pairs for training.

In [11]:
def turn_into_features(df, models):
    """Each battle supplies adjacent original and reversed examples."""
    models = list(models)
    index = {name: i for i, name in enumerate(models)}
    X = np.zeros((2 * len(df), len(models)))
    y = np.empty(2 * len(df), dtype=int)
    for i, row in enumerate(df.itertuples()):
        if row.winner not in ('model_a', 'model_b'):
            raise ValueError('Remove ties before constructing binary labels.')
        X[2*i, index[row.model_a]] += 1
        X[2*i, index[row.model_b]] -= 1
        X[2*i+1] = -X[2*i]
        y[2*i] = int(row.winner == 'model_a')
        y[2*i+1] = 1-y[2*i]
    return X, y
X, y = turn_into_features(selected_battles_no_ties, selected_models)
X.shape, y.shape

((50388, 20), (50388,))

In [12]:
grader.check("q4a")

q4a results: All test cases passed!

## **Question 4b**
Now that we have extracted out the features from the previous question, let's now dive into actually building the model. 

**Task:** 
Train the model with the features and labels created in Question 4a, and store the strengths, sorted in the order of scores in `results_df`.

In [13]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(fit_intercept=False, max_iter=2000, tol=1e-8).fit(X,y)
scores = model.coef_[0]
results_df = pd.DataFrame({'Model':selected_models,'Score':scores}).sort_values('Score',ascending=False).reset_index(drop=True)
results_df

,Model,Score
0,chatgpt-4o-latest,0.799297
1,gemini-1.5-pro-exp-0801,0.621566
2,gpt-4o-2024-05-13,0.469029
3,gpt-4o-mini-2024-07-18,0.440926
4,llama-3.1-405b-instruct,0.339940
5,claude-3-5-sonnet-20240620,0.335343
6,llama-3.1-70b-instruct,0.255561
7,gemini-1.5-pro-api-0514,0.233366
8,gpt-4-turbo-2024-04-09,0.226578
9,claude-3-opus-20240229,0.096847


In [14]:
grader.check("q4b")

q4b results: All test cases passed!

<!-- BEGIN QUESTION -->

## **Question 4c**

Let's think about an important aspect of our formulation.

**Task:** Answer the following question: Why we don't need an intercept for the logistic regression formulation above?

Only strength differences matter: adding a constant to every model score leaves all probabilities unchanged. With no intercept, reversing the competitors negates the logit, so P(A beats B) = 1 - P(B beats A); an intercept would instead introduce an A/B position advantage.

<!-- END QUESTION -->

# **Question 5: Confidence Intervals**

From the previous question, we were able to train the model and obtain the scores of the models. 

However, when comparing model scores, it's important to understand not just the average performance, but also how much uncertainty there is in our estimates. Our rankings are based on a finite sample of battles, and if we had collected a different set of match-ups, the resulting scores could be different. This sampling variability means that our estimated model strengths are subject to noise.

Bootstrapping is a powerful, intuitive way to assess this uncertainty without making strong assumptions about the underlying data. By repeatedly resampling our observed battles (with replacement) and retraining the model on each resampled dataset, we simulate what might have happened if we had observed a slightly different set of battles. For each resample, we get a new set of model scores. By looking at the distribution of these bootstrapped scores, we can estimate confidence intervals for each model's strength.

In short, bootstrapping helps us answer: "If we repeated this evaluation process many times, how much could each model's score vary just due to random chance in which battles we happened to observe?" This gives us a more honest sense of which differences in model scores are robust, and which might just be due to luck.

## **Question 5a**

Let's implement a function that returns these scores and confidence intervals after bootstrapping.

**Task:** 
* Bootstrap the samples to train a new logistic regression model.
* Store each set of coefficients (or learned model strengths).
* Compute the mean and percentiles (2.5th and 97.5th) to obtain the 95% confidence intervals.
* Return i) results_df, ii) mean_scores, iii) confidence_intervals.

An example outout of results_df is below.

<div style="text-align: center;">
  <img src="https://imgur.com/ZmC1PhW.png" alt="WarmupPairwisePlot" style="display: block; margin-left: auto; margin-right: auto; width: 80%;">
</div>

In [15]:
def get_bootstrapped_score(X, y, models, category_name='Overall', n_bootstrap=10):
    """Percentile intervals; resample whole battles, keeping reversed rows together."""
    X, y = np.asarray(X), np.asarray(y)
    if len(X) == 0 or n_bootstrap < 2:
        raise ValueError('Need nonempty data and at least two bootstrap replicates.')
    paired = (len(X) % 2 == 0 and np.allclose(X[0::2], -X[1::2])
              and np.array_equal(y[0::2], 1-y[1::2]))
    rng = np.random.RandomState(189)
    bootstrap_scores = []
    for _ in range(n_bootstrap):
        if paired:
            draws = rng.choice(len(X)//2, len(X)//2, replace=True)
            indices = np.column_stack((2*draws, 2*draws+1)).ravel()
        else:
            indices = rng.choice(len(X), len(X), replace=True)
        model = LogisticRegression(fit_intercept=False, C=1.0, max_iter=2000, tol=1e-8)
        model.fit(X[indices], y[indices])
        bootstrap_scores.append(model.coef_[0])
    bootstrap_scores = np.asarray(bootstrap_scores)
    mean_scores = bootstrap_scores.mean(axis=0)
    confidence_intervals = np.percentile(bootstrap_scores, [2.5, 97.5], axis=0)
    results_df = pd.DataFrame({'Model': list(models), 'Average Score': mean_scores,
        'Lower Bound': confidence_intervals[0], 'Upper Bound': confidence_intervals[1],
        'Category': category_name})
    results_df = results_df[['Model','Category','Average Score','Lower Bound','Upper Bound']]
    results_df['Rank'] = results_df.apply(
        lambda r: 1 + int((results_df['Lower Bound'] > r['Upper Bound']).sum()), axis=1)
    return results_df.sort_values('Average Score', ascending=False).reset_index(drop=True), mean_scores, confidence_intervals
results_df, mean_scores, confidence_intervals = get_bootstrapped_score(X, y, selected_models)
assert (confidence_intervals[0] <= confidence_intervals[1]).all()
# Ten replicates follow the exercise default; more are needed for stable tail estimates.
results_df

,Model,Category,Average Score,Lower Bound,Upper Bound,Rank
0,chatgpt-4o-latest,Overall,0.809331,0.751225,0.859767,1
1,gemini-1.5-pro-exp-0801,Overall,0.641859,0.592959,0.722792,2
2,gpt-4o-2024-05-13,Overall,0.448536,0.380879,0.518750,3
3,gpt-4o-mini-2024-07-18,Overall,0.446129,0.380793,0.505152,3
4,claude-3-5-sonnet-20240620,Overall,0.323002,0.287987,0.379526,5
5,llama-3.1-405b-instruct,Overall,0.318530,0.257002,0.383378,3
6,llama-3.1-70b-instruct,Overall,0.283847,0.166214,0.357517,5
7,gemini-1.5-pro-api-0514,Overall,0.227755,0.206094,0.265004,6
8,gpt-4-turbo-2024-04-09,Overall,0.206406,0.134927,0.261401,6
9,claude-3-opus-20240229,Overall,0.088240,0.028039,0.163394,9


In [16]:
grader.check("q5a")

q5a results: All test cases passed!

### Now let's visualize the intervals! *🧙*

In [17]:
results_df, mean_scores, confidence_intervals = get_bootstrapped_score(X, y, selected_models, n_bootstrap=25)
fig = go.Figure()

# Use the sorted values from results_df for plotting
fig.add_trace(go.Scatter(
    x=results_df["Model"],
    y=results_df["Average Score"],
    mode='markers',
    name='Model Scores',
    marker=dict(size=5, color='blue'),
    error_y=dict(
        type='data',
        array=results_df["Upper Bound"] - results_df["Average Score"],   # Upper error
        arrayminus=results_df["Average Score"] - results_df["Lower Bound"],  # Lower error
        visible=True
    )
))

fig.update_layout(
    title='Model Performance Scores with 95% Confidence Intervals (Sorted by Mean Score)',
    xaxis_title='Models',
    yaxis_title='Score',
    xaxis=dict(tickangle=45),
    height=500
)

fig.show()

## **Question 5b**

Now that we have confidence intervals, we can assign a rank to each model. We want the rank of model $i$ to represent the number of models that are **confidently better** than model $i$.

When we say model A is **confidently better** than model B, it will mean that model A's lower bound is still greater than model B's upper bound. Remember that greater rank means that there are more models that perform better than the current model.

**Task:**
Implement the `assign_rank` function below that assigns rank to the model.

In [18]:
def assign_rank(row, df=results_df):
    return 1 + int((df['Lower Bound'] > row['Upper Bound']).sum())
results_df['Rank'] = results_df.apply(lambda r: assign_rank(r, results_df), axis=1)
results_df = results_df.sort_values(['Rank', 'Average Score'], ascending=[True, False])
results_df

,Model,Category,Average Score,Lower Bound,Upper Bound,Rank
0,chatgpt-4o-latest,Overall,0.810109,0.738629,0.857499,1
1,gemini-1.5-pro-exp-0801,Overall,0.613718,0.529710,0.707100,2
2,gpt-4o-2024-05-13,Overall,0.464446,0.389938,0.524056,3
3,gpt-4o-mini-2024-07-18,Overall,0.453511,0.385672,0.505647,3
4,llama-3.1-405b-instruct,Overall,0.336491,0.269128,0.395795,3
5,claude-3-5-sonnet-20240620,Overall,0.326781,0.285986,0.382134,5
6,llama-3.1-70b-instruct,Overall,0.267160,0.158793,0.375386,5
8,gpt-4-turbo-2024-04-09,Overall,0.222658,0.136804,0.291161,5
7,gemini-1.5-pro-api-0514,Overall,0.232852,0.204662,0.267747,7
9,claude-3-opus-20240229,Overall,0.090081,0.029233,0.148005,9


In [19]:
grader.check("q5b")

q5b results: All test cases passed!

In [20]:
fig = plot_rank_heatmap(results_df)
fig.show()

<!-- END QUESTION -->

> **NOTICE BEFORE YOUR PROGRESS
(Q6 Data)**
> - If you accidentally modify `selected_battles_no_ties` in a way that breaks later parts, double check and **reset it** using the initial block of code you have placed.

# **Question 6: Category Leaderboards**

So far, we have computed overall model rankings using all available battles.
However, models may perform differently in specific categories, such as creativity, technical_accuracy, instruction_following, or math.
Now that we know how to get rankings, let's see what the leaderboards look like for certain categories.

Breaking this down, we want to do the following:

1. For each category we are interested in, filter the battles to only those belonging to a given category.
2. Compute bootstrapped confidence intervals for model strengths in that category.
3. Rank the models within that category.
4. Combine category-specific ranks with the overall leaderboard into a single DataFrame.

### Function Reference For Q6

Below is a summary of the functions you have already implemented that might be helpful. Remember that the **overall leaderboard** (used in Q6d) should be a DataFrame named `results_df` with a unique **`Model`** column and an overall **`Rank`**.

| Function | Inputs (types) | Output | One-liner purpose | Where you’ll use it |
|---|---|---|---|---|
| `turn_into_features` | `df_filtered: pd.DataFrame`, `selected_models: list[str]` | `X, y` | Build model–vs–model feature matrix `X` and labels `y` from filtered battles. | Q6a, Q6d |
| `get_bootstrapped_score` | `X`, `y`, `selected_models: list[str]`, `n_bootstrap: int` | `results_df, ci_low, ci_high` | Bootstrap model strengths; returns per-model scores + confidence intervals. | Q6a, Q6d |
| `assign_rank` | `row: pd.Series` (row of `results_df`) | `int` | Compute a row's rank from its score(s). | Q6a, Q6d |

## **Question 6a**

Let's start with the first step described above! 

Specifically, we will implement a general function that returns a DataFrame filtered to the battles belonging to a given mask, computes model scores using bootstrapping, assigns ranks, and finally returns the results sorted by rank. This function will allow us to conveniently obtain the score for any specific category.

---

**Task:**
Implement a function `get_category_results` that does the following:

- Takes in a DataFrame of battles, a boolean filter mask for a category, and a list of models.  
- Filters the battles to those in the category.
- Turns the filtered battles into features using `turn_into_features`
- Computes bootstrapped scores for the selected models using `get_bootstrapped_score`
- Assigns ranks based on model performance.  
- Returns a DataFrame sorted by ascending rank (best model first).  


**Parameters:**
- **`df` (pd.DataFrame)**  
  The full battles DataFrame. Each row corresponds to a single head-to-head battle between two models, along with metadata such as the category of the prompt (e.g., `"math"`, `"coding"`, `"writing"`).  

- **`filter_mask` (pd.Series[bool])**  
  A boolean array (same length as `df`) that indicates which rows to keep.  
  - Example: `filter_mask = (df["category"] == "math")` produces a Series of `True`/`False` values.  
  - When applied as `df.loc[filter_mask]`, only rows where the mask is `True` are kept.  
  - This lets us focus only on battles from a specific category.  

- **`selected_models` (list[str])**  
  A list of model names (strings) to evaluate and compare. The function will restrict bootstrapped scoring and ranking to this set.  
  - Example: `["gpt-4", "llama-2", "claude-3"]`.  

- **`n_bootstrap` (int, optional, default = 25)**  
  The number of bootstrap resamples to use when estimating model scores. Larger values give more stable estimates but take longer to compute.

In [21]:
def get_category_results(df, filter_mask, selected_models, category_name='Overall', n_bootstrap=25):
    filtered_df = df.loc[filter_mask]
    X_cat,y_cat = turn_into_features(filtered_df,selected_models)
    result,_,_ = get_bootstrapped_score(X_cat,y_cat,selected_models,category_name,n_bootstrap)
    result['Rank'] = result.apply(lambda r:assign_rank(r,result),axis=1)
    return result.sort_values(['Rank','Average Score'],ascending=[True,False])

In [22]:
grader.check("q6a")

q6a results: All test cases passed!

## **Question 6b**

We computed the score for each of the categories in the previous question. Ultimately, we want to compute the intervals for model strengths in these new categories like we originally did before. To achieve this, we first need to extract out the relevant characteristic of each of the battles (whether it is creative, has certain technical accuracy, etc.).

**Task:**
Using the `selected_battles_no_ties` DataFrame, create four new boolean columns that indicate whether each battle belongs to a given category:
* creative
* technical_accuracy
* instruction_following
* math

These columns should be derived from the nested dictionary in the `category_tag `column.
We are going to make a copy first to avoid pandas SettingWithCopyWarning.

**Hint:** Each of these categories is stored inside a specific subkey (e.g., "criteria_v0.1" or "math_v0.1") within `category_tag`.

In [23]:
selected_battles_no_ties = selected_battles_no_ties.copy()
for col,group,key in [('creative','criteria_v0.1','creativity'),('technical_accuracy','criteria_v0.1','technical_accuracy'),
                      ('instruction_following','if_v0.1','if'),('math','math_v0.1','math')]:
    selected_battles_no_ties[col] = selected_battles_no_ties['category_tag'].apply(lambda x:bool(x[group][key]))

In [24]:
# Memory cleanup: delete large dataframes no longer needed
del battles, selected_battles
import gc
gc.collect()

595

In [25]:
grader.check("q6b")

q6b results: All test cases passed!

## **Question 6c**

Now that obtained the relevant characteristic of each battle, let's try to define a filter that extracts out the battles we want for each category. 

**Task:**
Define the category filters for each of the categories.
Specifically, using the `selected_battles_no_ties` DataFrame, create a dictionary called `category_filters` that maps each category name to a boolean mask selecting only the battles in that category.

Your dictionary should include filters for:
*  'english' (battles where language is "English")
*  'coding' (battles where is_code is True)
*  'creative' (battles where creative is True)
*  'instruction_following' (battles where instruction_following is True)
*  'math' (battles where math is True)
*  'technical_accuracy' (battles where technical_accuracy is True)

In [26]:
category_filters = {'english':selected_battles_no_ties['language'].eq('English'),
                    'coding':selected_battles_no_ties['is_code'].eq(True)}
category_filters.update({col:selected_battles_no_ties[col] for col in
    ['creative','instruction_following','math','technical_accuracy']})

In [27]:
grader.check("q6c")

q6c results: All test cases passed!

## **Question 6d**

Now that we have all the filters defined, let's compute the **per-category leaderboards** and combine them with the **overall leaderboard** in tidy data format.

**Task:**
Compute per-category leaderboards using `get_category_results` and combine them into a single tidy DataFrame.

1. **Overall leaderboard:** Use `get_category_results` with a mask that includes all battles (all `True` values) to generate the overall leaderboard with category name "Overall".

2. **Per-category leaderboards:** For each category in `category_filters`, use `get_category_results` to build a DataFrame containing `Model`, `Category`, `Average Score`, `Lower Bound`, `Upper Bound`, and `Rank`.

3. **Tidy format combination:** Concatenate all category results into a single DataFrame with columns `['Model', 'Rank', 'Category']` where:
   - Each row represents one model's performance in one category
   - The `Category` column identifies which category the rank belongs to (e.g., "Overall", "english", "coding", etc.)

4. **Sort final table:** Sort each category dataframe by Rank then sort the entire dataframe by Category (both ascending).

**Example Output:**

| Model               | Rank | Category              |
|---------------------|-----:|----------------------:|
| chatgpt-4o-latest   | 1    | Overall               |
| gemini-1.5-pro-exp-0801 | 2    | Overall               |
| gpt-4o-2024-05-13   | 3    | Overall               |
| chatgpt-4o-latest   | 1    | english               |
| gemini-1.5-pro-exp-0801 | 2    | english               |
| gpt-4o-2024-05-13   | 4    | english               |
| chatgpt-4o-latest   | 1    | coding                |
| gpt-4o-2024-05-13   | 2    | coding                |

This is in the same tidy format as the previous part of the homework.

In [28]:
overall_mask = pd.Series(True,index=selected_battles_no_ties.index)
tables = [get_category_results(selected_battles_no_ties,overall_mask,selected_models)]
for category,mask in category_filters.items():
    tables.append(get_category_results(selected_battles_no_ties,mask,selected_models,category))
category_results_df = pd.concat(tables,ignore_index=True).sort_values(['Rank','Category']).reset_index(drop=True)
category_results_df

,Model,Category,Average Score,Lower Bound,Upper Bound,Rank
0,chatgpt-4o-latest,Overall,0.810109,0.738629,0.857499,1
1,chatgpt-4o-latest,coding,0.846046,0.640699,1.106357,1
2,claude-3-5-sonnet-20240620,coding,0.556164,0.434618,0.663007,1
3,gemini-1.5-pro-exp-0801,coding,0.447907,0.323715,0.643751,1
4,chatgpt-4o-latest,creative,0.850428,0.736750,0.942640,1
...,...,...,...,...,...,...
135,llama-3-8b-instruct,creative,-0.773598,-0.936052,-0.600051,17
136,llama-3-8b-instruct,technical_accuracy,-0.731453,-0.837006,-0.581220,17
137,llama-3-8b-instruct,instruction_following,-0.822276,-0.978613,-0.664545,18
138,llama-3-8b-instruct,math,-1.222682,-1.465808,-0.930822,18


In [29]:
grader.check("q6d")

q6d results: All test cases passed!

Amazing! Now let's plot the heatmap showing rank across the categories 🧙.

In [30]:
fig = plot_rank_heatmap(category_results_df, title="Model Rankings by Category")
fig.show()

**Something to ponder upon:** We see model rankings can change a lot depending on the type of question being asked. Sometimes these make sense, like how deepseek coder gets much higher rankings on coding problems, but sometimes it isn't clear why one model does better than another. Especially for things like creative tasks, why do people like Gemini 1.5 so much more than Claude 3.5?

# **Question 7: Ranking Influences**

One thing that has been known to affect user preference is response length: people (and LLM's) tend to prefer longer answers. A recurring observation in human grading and UX is that **longer responses are often preferred**. For example, analyses from the SAT essay reported that **essay length strongly correlated with higher scores—even when errors were present** ([New York Times, 2005](https://www.nytimes.com/2005/05/04/education/sat-essay-test-rewards-length-and-ignores-errors.html)). 

In the context of LLM evaluations, this motivates a core question: **does response length systematically tilt battle outcomes and model rankings?**


Let's investigate whether length plays a role in model rankings. First let's do some quick analysis on the response length per model

## **Question 7a**

We want to analyze whether **response length (in tokens)** is related to model rankings.

In `per_model_battles` (which is what you would implement in Q7b), the **`conversation`** column contains, for each row, a *single exchange* between a user and a model (one battle). It is represented as a **list of message dictionaries**. These dictionaries are representing a full exchange between a user and a model in a single battle. The number of turns is `len(row['conversation']/2`.

**Each message dictionary contains (as provided):**
1. `"content"` – the text of the message  
3. `"role"` – either `"user"` or `"assistant"`. In our question we will be focusing on `"assistant"`

---

#### What exactly is `conv`?

For this question, assume your function will receive **`conv`**, which is a **dictionary** with a single key `"conversation"` mapping to a **list of message dictionaries**:

```python
conv_example = {
    "conversation": [
        {"role": "user", "content": "How do I sum a list in Python?"},
        {"role": "assistant", "content": "Use the built-in function: sum(your_list)."}
    ]
}
# conv_example["conversation"]  -> list of message dicts, ordered by turns
# Each dict has:
#   - "role": "user" or "assistant"
#   - "content": str (message text)

**Task:**
Implement a function `calculate_response_length` that, given a conversation `conv`, returns the total number of GPT-2 tokens in the concatenation of all **assistant messages** in that conversation.

**Requirements:**

1. Use the tiktoken library with the "gpt2" encoding. Import tiktoken.

2. Concatenate only the assistant messages and count tokens.

3. Call `enc.encode(..., disallowed_special=())` to allow all special tokens (avoids ValueError, e.g., for <|endoftext|>).

4. Concatenate all assistant role messages separated by two newlines ("\n\n") before counting tokens (join each them by this).

In [31]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
def calculate_response_length(conv):
    text = '\n\n'.join(m['content'] or '' for m in conv['conversation'] if m['role'] == 'assistant')
    return len(enc.encode(text, disallowed_special=()))

In [32]:
grader.check("q7a")

q7a results: All test cases passed!

## **Question 7b**

In the previous question (Q7a), you wrote a function to compute the **token length** of a model's reply from a conversation. We’ll now **reshape** the battle-level data so that each row corresponds to a **single model’s response in a single battle** (instead of one row per battle).

In the `selected_battles_no_ties` DataFrame, each row represents a battle between two models, with:

*   conversation_a = the conversation for model_a in that battle
*   conversation_b = the conversation for model_b in that battle


To analyze response length per model, we would want a table where each row corresponds to a single model's response in a single battle (rather than one row per battle).

Specifically, our goal is to turn each battle row into **two rows** (tidy format):
1. one for `model_a` using `conversation_a`
2. one for `model_b` using `conversation_b`

**Task:**
Using the function defined in Question 7a, create a DataFrame named `per_model_battles ` with columns:

1. **`conversation`** — the list of message dicts for that model’s side of the battle  
2. **`model`** — the model name  
3. **`response_length`** — integer token count of all assistant messages concatenated (computed via **`calculate_response_length`** from Q7a)
> **Hint:** `pd.concat` might be handy for stacking the A-side and B-side tables into one.  
> Docs: https://pandas.pydata.org/docs/reference/api/pandas.concat.html

In [33]:
battles_a = selected_battles_no_ties[['model_a', 'conversation_a']].rename(
    columns={'model_a':'model', 'conversation_a':'conversation'})
battles_b = selected_battles_no_ties[['model_b', 'conversation_b']].rename(
    columns={'model_b':'model', 'conversation_b':'conversation'})
per_model_battles = pd.concat([battles_a, battles_b], ignore_index=True)
per_model_battles['response_length'] = per_model_battles.apply(calculate_response_length, axis=1)
per_model_battles = per_model_battles[['conversation','model','response_length']]
per_model_battles.head()

,conversation,model,response_length
0,[{'content': 'Is there any Artificial Superint...,gemma-2-9b-it,364
1,[{'content': 'チャットアプリのフロントエンドとバックエンドをそれぞれ作成して。...,gpt-4o-2024-05-13,1494
2,"[{'content': '其味无穷无', 'num_tokens': 7, 'role':...",gemma-2-27b-it,172
3,"[{'content': '1번글 옷은 단순히 물리적 보호를 넘어서, 우리의 정체성,...",claude-3-opus-20240229,2595
4,"[{'content': 'if I had 3 aples but ate 2, how ...",claude-3-5-sonnet-20240620,78


In [34]:
grader.check("q7b")

q7b results: All test cases passed!

In [35]:
# Let's take a look at the structure
# print(per_model_battles['conversation'].iloc[0])

Let's plot the response length for each model ordered by their rank and fit a trendline to see if there is any relation between rank and length.

In [36]:
model_lineup = results_df.sort_values("Rank")['Model'].tolist()
avg_lengths = per_model_battles.groupby("model")["response_length"].mean().reset_index()
avg_lengths["model"] = pd.Categorical(avg_lengths["model"], categories=model_lineup, ordered=True)
avg_lengths = avg_lengths.sort_values("model").reset_index(drop=True)

# Add a numeric rank column for trendline fitting
avg_lengths["rank"] = avg_lengths.index + 1  # 1 = best, etc.

# Fit a linear trendline (polyfit) to the response length vs. rank
z = np.polyfit(avg_lengths["rank"], avg_lengths["response_length"], 1)
p = np.poly1d(z)
trendline = p(avg_lengths["rank"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=avg_lengths["model"],
    y=avg_lengths["response_length"],
    mode='lines+markers',
    name='Avg Response Length'
))

fig.add_trace(go.Scatter(
    x=avg_lengths["model"],
    y=trendline,
    mode='lines',
    name='Trendline',
    line=dict(dash='dash', color='red')
))

fig.update_layout(
    title="Average Response Length of Models (with Trendline)",
    xaxis_title="Model (sorted by performance)",
    yaxis_title="Average Response Length",
    xaxis_tickangle=45,
    yaxis=dict(range=[0, max(avg_lengths["response_length"].max(), trendline.max()) * 1.05])  # y-axis starts at 0
)

fig.show()

# **Question 8: Style Control**

It looks like there is a trend: models with shorter responses tend to be ranked lower. While not a perfect analysis, if it could be true that people are preferring models which generate longer responses regardless of their other capabilities, then it would be useful to create a leaderboard which is *length agnostic*. Meaning, creating leaderboard model scores that control for certain stylistic properties of responses.

So how can we control for these stylistic factors in our model rankings?

In this question, you will implement a style feature ranking pipeline, starting with length as the only style feature.

1. Each row in `selected_battles_no_ties` represents a battle between model_a and model_b.

2. Each battle contains `conv_metadata` with pre-computed style metrics, such as bold text counts, header counts, list counts, and token counts for each side.

3. In the earlier question, each battle was converted to pairwise feature, where 1 denoted the model it belongs to, and -1 for the model that it was battling against. Now, our goal is to build on that data, including both the model identity indicators and the chosen style features.

## **Question 8a**

We want to add style features other than length that will give us style feature aware ranks.


**Task:**
Implement the function `add_style_features` that reads stylistic metrics from `conv_metadata` between each model (model_a, model_b) for count of bold, header, list, and assistant tokens. Then, store the normalized differences in columns (with the designated names):

1. style_bold_count
2. style_header_count
3. style_list_count
4. style_sum_assistant_tokens

The normalized differences would be following the formulation below:

$$
\text{normdiff}(a, b) =
\begin{cases}
0 & \text{if } a + b = 0 \\[6pt]
\dfrac{a - b}{a + b} & \text{otherwise}
\end{cases}
$$


⚠️ **Important note**: Make sure you are not mutating the original DataFrame passed into your function. Work on a copy (df.copy()) and return that new DataFrame with the added columns.

Let's take a took at `conv_metadata`. Essentially, we will be stacking up these elements for each style counts.

In [37]:
selected_battles_no_ties['conv_metadata'].iloc[0]

{'bold_count_a': {'**': 5, '__': 0},
 'bold_count_b': {'**': 0, '__': 0},
 'context_a_tokens': 222,
 'context_b_tokens': 230,
 'header_count_a': {'h1': 0, 'h2': 0, 'h3': 0, 'h4': 0, 'h5': 0, 'h6': 0},
 'header_count_b': {'h1': 0, 'h2': 0, 'h3': 0, 'h4': 0, 'h5': 0, 'h6': 0},
 'list_count_a': {'ordered': 0, 'unordered': 5},
 'list_count_b': {'ordered': 0, 'unordered': 0},
 'sum_assistant_a_tokens': 335,
 'sum_assistant_b_tokens': 287,
 'sum_user_tokens': 14,
 'turns': 2}

In [38]:
def add_style_features(df):
    df = df.copy()
    def normdiff(a, b):
        return (a-b)/(a+b) if a+b else 0.0
    for key in ['bold_count', 'header_count', 'list_count', 'sum_assistant_tokens']:
        def difference(m):
            if key == 'sum_assistant_tokens':
                a, b = m['sum_assistant_a_tokens'], m['sum_assistant_b_tokens']
            else:
                a, b = sum(m[key+'_a'].values()), sum(m[key+'_b'].values())
            return normdiff(a, b)
        df['style_'+key] = df['conv_metadata'].apply(difference)
    return df
selected_battles_no_ties = add_style_features(selected_battles_no_ties)

In [39]:
grader.check("q8a")

q8a results: All test cases passed!

<!-- BEGIN QUESTION -->

## **Question 8a Free Response Question**

We’ve now added stylistic features to each model comparison.  

**Answer the following question**

```otter
QUESTION: How can integrating these features into the ranking pipeline create the effect of a “length-controlled” leaderboard, and why might this adjustment be useful?  

Think about whether raw win/loss outcomes fully capture model quality, or whether stylistic inflation (e.g., longer answers, formatting tricks) can bias rankings.
```

The regression includes both model identity differences and normalized style differences. Model coefficients therefore describe comparisons at equal values of the included style covariates, rather than attributing every preference for longer or more formatted responses to model identity. This helps separate observed style associations from model rankings, but it does not establish causal effects or remove unmeasured confounding.

<!-- END QUESTION -->

## **Question 8b**

Let's try to visualize and formulate the features we defined in the previous question in a neat way that we can see the direct relationship between the model battles and the style features.

We now want a training table where each battle produces two rows, one for each ordering of the competitors (A→B and B→A).
In other words, each row in the dataframe creates two entries in the new table.

**Task:**
Implement a function that creates the table described above. Each row should encode:
1. The model identity vector **X** (+1 at the selected model in the row, −1 at its opponent, 0 elsewhere)
2. The outcome y (win, lose)
3. Set of style covariates capturing A vs B normalized differences (e.g., length)

**NOTE:** Ensure that features are also antisymmetric, meaning they flipping the order of model should also flip the sign of each style feature.

Below is an example of the desired table.

| question_id                         | X                                           | y | direction | style_bold_count | style_header_count | style_list_count | style_sum_assistant_tokens |
|--------------------------------------|----------------------------------------------|---|-----------|------------------|--------------------|------------------|----------------------------|
| e8fe7c9f75ab4e528367cc7de625c475     | [0, 0, 0, 0, 0, 1, ...]  | 0 | A->B      | 1.0              | 0.0                | 1.0              | 0.07717                    |
| e8fe7c9f75ab4e528367cc7de625c475     | [0, 0, 0, 0, 0, -1 ...] | 1 | B->A      | -1.0             | -0.0               | -1.0             | -0.07717                   |

In [40]:
def make_pairwise_feature_df(df, models, style_feature_cols):
    X, y = turn_into_features(df, models)
    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        for offset, sign, direction in [(0, 1, 'A->B'), (1, -1, 'B->A')]:
            record = {'question_id': row['question_id'], 'X': X[2*i+offset],
                      'y': y[2*i+offset], 'direction': direction}
            record.update({col: sign*float(row[col]) for col in style_feature_cols})
            records.append(record)
    return pd.DataFrame(records, columns=['question_id','X','y','direction']+list(style_feature_cols))
style_feature_cols = ['style_bold_count','style_header_count','style_list_count','style_sum_assistant_tokens']
pairwise_feature_df = make_pairwise_feature_df(selected_battles_no_ties, selected_models, style_feature_cols)
pairwise_feature_df.head(2)

,question_id,X,y,direction,style_bold_count,style_header_count,style_list_count,style_sum_assistant_tokens
0,e8fe7c9f75ab4e528367cc7de625c475,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...",0,A->B,1.0,0.0,1.0,0.07717
1,e8fe7c9f75ab4e528367cc7de625c475,"[-0.0, -0.0, -0.0, -0.0, -0.0, -1.0, -0.0, -0....",1,B->A,-1.0,-0.0,-1.0,-0.07717


In [41]:
grader.check("q8b")

q8b results: All test cases passed!

## **Question 8c**

Amazing! 🎉 Now that we've built our pairwise identity matrix X (which model is battling which) and our style feature matrix X_style (how A and B differ stylistically), lets's combine these two so that our logistic model can learn:

1. The intrinsic strength of each model (controlling for style)
2. The influence of each style feature on the outcome

**Task:**
Now, implement the function `get_sc_category_results` that:
1. Stacks these two matrices into one design matrix X_with_style, so the model can learn both intrinsic model strengths and style effects simultaneously.
2. Uses `get_bootstrapped_score` to get the ranking results.
3. Returns the results sorted by rank in ascending order. All style features should be assigned a rank of -1

In [42]:
def get_sc_category_results(df, selected_models, filter_mask=None,
                            category_name='Overall w/ Style Control', n_bootstrap=10,
                            style_features=style_feature_cols):
    subset = df if filter_mask is None else df.loc[filter_mask]
    pairwise = make_pairwise_feature_df(subset, selected_models, style_features)
    X_with_style = np.column_stack([np.stack(pairwise['X']), pairwise[style_features].to_numpy()])
    result, _, _ = get_bootstrapped_score(X_with_style, pairwise['y'].to_numpy(),
        list(selected_models)+list(style_features), category_name, n_bootstrap)
    model_rows = result[result['Model'].isin(selected_models)]
    result['Rank'] = result.apply(lambda row: assign_rank(row, model_rows)
        if row['Model'] in selected_models else -1, axis=1)
    return result.sort_values(['Rank','Average Score'], ascending=[True,False]).reset_index(drop=True)
results_df_style_control = get_sc_category_results(selected_battles_no_ties, selected_models)
combined_results_df = pd.concat([results_df, results_df_style_control], ignore_index=True)
fig = plot_rank_heatmap(combined_results_df, title='With and Without Style Control', selected_models=selected_models)
fig.show()

In [43]:
grader.check("q8c")

q8c results: All test cases passed!

### Look at the impact of style

Now let's visualize the style feature scores with confidence intervals using our premade `plot_style_features` in `plotting_utils.py`.

In [44]:
# plot style feature coefficients
fig = plot_style_features(results_df_style_control, selected_models)
fig.show()

Here we see that length matters a LOT (in fact this coefficient is higher than the actual model coefficients), while things like bold and lists don't matter as much.

## **Question 8d**

Let's add the stylistic features to our computation of per-category leaderboards.

**Goal:** Extend your category leaderboards from Q7d to the style-controlled setting.

**Note:** This mirrors the merge pattern you used in Q7d. Reuse that approach, but start from the style-controlled baseline (not the plain baseline).

**Task:**
Now, re-using the function `get_sc_category_results` and `category_filters` that was previously defined, define `category_style_control_results_df` that has all the bound scores for the other categories (english, coding, creative, instruction_following, math, technical_accuracy). Specifically,

1. Using `get_sc_category_results` and the provided category_filters, compute per-category results. 

3. Name the final DataFrame category_stle_control_results_df and sort by "Rank" ascending. This should be the same structure as the previous result `combined_results_df`.

In [45]:
tables = [results_df_style_control]
for category,mask in category_filters.items():
    tables.append(get_sc_category_results(selected_battles_no_ties,selected_models,
        filter_mask=mask,category_name=category+' w/ Style Control'))
category_style_control_results_df = pd.concat(tables,ignore_index=True).sort_values(['Rank','Category']).reset_index(drop=True)

In [46]:
grader.check("q8d")

q8d results: All test cases passed!

In [47]:
fig = plot_rank_heatmap(category_style_control_results_df, title="With Style Control")
fig.show()

Now let's look at the delta in rankings (shift of ranking) when we appy style control across all these categories.

In [48]:
style_control_models = category_style_control_results_df[
    category_style_control_results_df['Model'].isin(selected_models)
].copy()

baseline_models = category_results_df[
    category_results_df['Model'].isin(selected_models)
].copy()
style_pivot = style_control_models.pivot(index='Model', columns='Category', values='Rank')
baseline_pivot = baseline_models.pivot(index='Model', columns='Category', values='Rank')

# Create mapping between style control and baseline categories
category_mapping = {}
for style_cat in style_pivot.columns:
    baseline_cat = style_cat.replace(" w/ Style Control", "")
    if baseline_cat in baseline_pivot.columns:
        category_mapping[style_cat] = baseline_cat

print(f"Category mapping: {category_mapping}")

if not category_mapping:
    print("No matching categories found between style control and baseline results")
else:
    common_models = list(set(style_pivot.index) & set(baseline_pivot.index))
    print(f"Comparing {len(common_models)} models across {len(category_mapping)} categories")
    style_aligned = pd.DataFrame(index=common_models)
    baseline_aligned = pd.DataFrame(index=common_models)
    for style_cat, baseline_cat in category_mapping.items():
        style_aligned[baseline_cat] = style_pivot.loc[common_models, style_cat]
        baseline_aligned[baseline_cat] = baseline_pivot.loc[common_models, baseline_cat]
    
    # Compute rank deltas (baseline - style_control)
    delta_data = baseline_aligned - style_aligned
    if 'Overall' in delta_data.columns:
        delta_data = delta_data.drop(columns=['Overall'])

    heatmap_z = delta_data.values
    heatmap_x = delta_data.columns.tolist()  # Categories
    heatmap_y = delta_data.index.tolist()    # Models
    avg_delta = delta_data.mean(axis=1).sort_values(ascending=False)
    delta_data_sorted = delta_data.loc[avg_delta.index]
    heatmap_z = delta_data_sorted.values
    heatmap_y = delta_data_sorted.index.tolist()

    annotations = []
    for i, model in enumerate(heatmap_y):
        for j, category in enumerate(heatmap_x):
            value = heatmap_z[i][j]
            if not pd.isna(value):
                annotations.append(dict(x=category,y=model,text=f"{int(value)}",showarrow=False,font=dict(color="black" if abs(int(value)) < 2 else "white", size=12)))
    n_models = len(heatmap_y)
    height = max(400, n_models * 30)
    
    fig = go.Figure(data=go.Heatmap(z=heatmap_z,x=heatmap_x,y=heatmap_y,colorscale="RdBu",colorbar=dict(title="Rank Delta (Baseline - Style Control)"),zmid=0))
    fig.update_layout(
        title="Delta in Model Rankings With Style Control (Category-Specific)",
        xaxis_title="Category",
        yaxis_title="Model",
        yaxis_autorange="reversed",
        annotations=annotations,
        height=height
    )
    fig.show()

Category mapping: {'Overall w/ Style Control': 'Overall', 'coding w/ Style Control': 'coding', 'creative w/ Style Control': 'creative', 'english w/ Style Control': 'english', 'instruction_following w/ Style Control': 'instruction_following', 'math w/ Style Control': 'math', 'technical_accuracy w/ Style Control': 'technical_accuracy'}
Comparing 20 models across 7 categories


Here we can quickly see which models are "Style hacking" - formatting their responses nicely but not necesarily being more capable models. It looks like gpt-4o-mini, llama-3.1-70b-instruct, llama-3.1-8b-instruct see a consistent drop in rankings while claude 3.5 sonnet, gemma-2-27b, and claude-3-haiku see a consistent rise in rankings.

# **Question 9: Finding New Style Influences**

Earlier, we saw how model preference differs by looking at structural style features in model outputs (e.g., bold text count, header count, list count, token length).
Now let's see if we can find new style features by inspecting the model responses to understand differences in models.
Let's inpsect 🔍 the text ourselves for stylistic signals associated with wins. Your goal is to analyze assistant responses and identify phrases that differentiate winning from losing replies. This helps surface style features we might add to our ranking model later.

### Helper Functions
Function to turn a conversations into plain text:
*   convert_conversation_to_string
*   convert_asst_conversation_to_string

Function that compares two text with n-gram TF-IDF:
*   tfidf_phrase_diff

In [49]:
def convert_conversation_to_string(conv):
  ret = ""
  for i in conv:
    if i['role'] == 'user':
      ret += "User: " + i['content'] + "\n\n"
    else:
      ret += "Assistant: " + i['content'] + "\n\n"
  return ret

def convert_asst_conversation_to_string(conv):
  ret = ""
  for i in conv:
    if i['role'] == 'assistant':
      ret += i['content'] + "\n\n"
  return ret

In [50]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def is_number_phrase(phrase):
    # Remove phrases that are only numbers or contain only numbers and spaces/punctuation
    # Also remove phrases that are just a number or start/end with a number
    return bool(re.fullmatch(r"[\d\s\W]+", phrase)) or bool(re.search(r"\b\d+\b", phrase))

def tfidf_phrase_diff(str_list_a, str_list_b, name_a="A", name_b="B", top_n=30, max_features=1000):
    """
    Compute distinguishing ngram tfidf phrases between two sets of strings.
    Returns two DataFrames: one for phrases more common in A, one for B.
    """
    all_texts = str_list_a + str_list_b
    labels = [name_a] * len(str_list_a) + [name_b] * len(str_list_b)
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english', ngram_range=(2,4))
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    feature_names = vectorizer.get_feature_names_out()
    n = len(str_list_a)
    tfidf_a = tfidf_matrix[:n]
    tfidf_b = tfidf_matrix[n:]
    mean_a = np.asarray(tfidf_a.mean(axis=0)).flatten()
    mean_b = np.asarray(tfidf_b.mean(axis=0)).flatten()
    # Top phrases for A
    a_scores = mean_a - mean_b
    top_a_indices = np.argsort(a_scores)[::-1]
    top_a_phrases = []
    for i in top_a_indices:
        phrase = feature_names[i]
        if not is_number_phrase(phrase):
            top_a_phrases.append((phrase, mean_a[i], mean_b[i]))
        if len(top_a_phrases) >= top_n:
            break
    # Top phrases for B
    b_scores = mean_b - mean_a
    top_b_indices = np.argsort(b_scores)[::-1]
    top_b_phrases = []
    for i in top_b_indices:
        phrase = feature_names[i]
        if not is_number_phrase(phrase):
            top_b_phrases.append((phrase, mean_b[i], mean_a[i]))
        if len(top_b_phrases) >= top_n:
            break
    df_a = pd.DataFrame(top_a_phrases, columns=["phrase", f"{name_a}_tfidf", f"{name_b}_tfidf"])
    df_b = pd.DataFrame(top_b_phrases, columns=["phrase", f"{name_b}_tfidf", f"{name_a}_tfidf"])
    return df_a, df_b

In [51]:
# Example usage:
model = "llama-3.1-70b-instruct"
llama_battles = selected_battles_no_ties[
    (selected_battles_no_ties['model_a'] == model) | (selected_battles_no_ties['model_b'] == model)
].copy()

llama_battles.loc[:, "model_a_conversation_string"] = llama_battles["conversation_a"].apply(convert_asst_conversation_to_string)
llama_battles.loc[:, "model_b_conversation_string"] = llama_battles["conversation_b"].apply(convert_asst_conversation_to_string)
llama_battles = llama_battles[llama_battles['language'] == 'English']

str_list_a = llama_battles.apply(lambda x: x["model_a_conversation_string"] if x["model_a"] == model else x["model_b_conversation_string"], axis=1).tolist()
str_list_b = llama_battles.apply(lambda x: x["model_b_conversation_string"] if x["model_a"] == model else x["model_a_conversation_string"], axis=1).tolist()
# print(str_list_a)

df_a, df_b = tfidf_phrase_diff(str_list_a, str_list_b, model, "others")

print(f"Top {model} phrases:")
# display(df_a)
print(f"Top others phrases:")
# display(df_b)

Top llama-3.1-70b-instruct phrases:
Top others phrases:


# **Question 9a: Key Phrases**

Let's analyze which phrases inherent in the text might be related to the win or lose of the battles. This is a similar idea to VibeCheck except (1) instead of comparing moel pairs we care comparing winning vs losing models and (2) instead of using LLM's to propose and validate vibes, we are going to be relying on keyword matching. 

Using the helper functions provided (`convert_asst_conversation_to_string`, `tfidf_phrase_diff`) and the reference example as guidance, implement a winning-vs-losing phrase analysis for assistant responses.




**Task:** 

1. From selected_battles_no_ties, keep only rows in English.

2. For each battle, extract assistant-only text using convert_asst_conversation_to_string.

3. Build winning_responses: one assistant-only string for the winning side of each battle.

4. Build losing_responses: one assistant-only string for the losing side of each battle.

5. Use tfidf_phrase_diff to compare the two lists and construct df_win and df_lose. (we have set this up for you)

6. Display the results to see the phrases most associated with winning vs. losing. (we have set this up for you)

For your reference, a sample subset of outputs is shown below.

### Example: Top phrases in *winning* responses
| phrase              | winning_tfidf | losing_tfidf |
|---------------------|---------------|--------------|
| let break           | 0.0127        | 0.0106       |
| step step           | 0.0146        | 0.0126       |\
| ... |...       | ...     |

### Example: Top phrases in *losing* responses
| phrase                 | losing_tfidf | winning_tfidf |
|------------------------|--------------|----------------|
| let know               | 0.0278       | 0.0190         |
| sorry assist           | 0.0054       | 0.0003         |
| ...     | ...      | ...  |

In [52]:
selected_battles_english = selected_battles_no_ties.loc[selected_battles_no_ties['language'].eq('English')].copy()
for side in ['a', 'b']:
    selected_battles_english['text_'+side] = selected_battles_english['conversation_'+side].apply(convert_asst_conversation_to_string)
winning_responses = np.where(selected_battles_english['winner'].eq('model_a'),
    selected_battles_english['text_a'], selected_battles_english['text_b']).tolist()
losing_responses = np.where(selected_battles_english['winner'].eq('model_a'),
    selected_battles_english['text_b'], selected_battles_english['text_a']).tolist()
df_win, df_lose = tfidf_phrase_diff(winning_responses, losing_responses, 'winning', 'losing')
display(df_win, df_lose)

,phrase,winning_tfidf,losing_tfidf
0,let break,0.012699,0.010620
1,united states,0.009846,0.007783
2,step step,0.014593,0.012553
3,said voice,0.004666,0.002649
4,crucial role,0.004945,0.003120
5,worth noting,0.006328,0.004511
6,important note,0.013610,0.012052
7,problem solving,0.005685,0.004168
8,don know,0.005111,0.003595
9,tenths place,0.002560,0.001138


,phrase,losing_tfidf,winning_tfidf
0,let know,0.027772,0.019045
1,sorry assist,0.005370,0.000267
2,feel comfortable,0.006051,0.001076
3,assist request,0.004613,0.000339
4,sorry assist request,0.004459,0.000191
5,let know like,0.010424,0.006354
6,know like,0.010616,0.006771
7,know questions,0.008371,0.005411
8,let know questions,0.008371,0.005411
9,provide information,0.006867,0.004291


In [53]:
grader.check("q9a")

q9a results: All test cases passed!

## Feature Exploration

One thing we see from the TF-IDF results is that "i'm sorry" or "i apologize" appear often in losing models - when looking through these conversations you will see that these are often instances of **refusal**: where the model refuses to answer the question beacuse it violates ethical guidelines or is out of its domain of knowledge. Now let's turn this into a style feature to measure its impact on accuracy. 


Let's try capturing whether the assistant on side A which apologizes more than side B in a battle by calculating the normalized sorry_count_diff. Here we will just have a binary 1/0 for each conversation indicating if it contains or does not contain the word "sorry".

In [54]:
def count_phrase_diff(row, phrase=["step by step"]):
    step_by_step_a = False
    step_by_step_b = False
    for i in row["conversation_a"]:
        if i["role"] == "assistant" and any([p in i["content"].lower().replace("-", " ") for p in phrase]):
            step_by_step_a = True
    for i in row["conversation_b"]:
        if i["role"] == "assistant" and any([p in i["content"].lower().replace("-", " ") for p in phrase]):
            step_by_step_b = True
    return int(step_by_step_a) - int(step_by_step_b)


selected_battles_no_ties.loc[:, "refusal_count"] = selected_battles_no_ties.apply(
    lambda row: count_phrase_diff(row, ["sorry", "apologize"]),
    axis=1
)

<!-- BEGIN QUESTION -->

# **Question 9b: Discover Some Immaculate Vibes**

**Task:** 
Now, just like the `refusal_count` feature we created above, implement new functions that can extract any stylistic features from the conversations.  
- Define a function that computes the normalized difference for a feature of your choice (e.g., presence of certain phrases, punctuation, formatting). You can check multiple different phrases if you want, they just to have a common "theme" - similar to the last problem of the previous part of this homework.   
- Apply this function to each row in the dataset.  
- Store the results in a new column of `selected_battles_no_ties["YOUR_FEATURE"]`. 
- Plot the change in ranking and style coefficients and the existing style features along with your custom feature. **To get full points, your feature need to get a higher coefficient (Average Score) than `style_header_count`**. It is okay if the confidence intervasls overlap. 
- You cannot use a feature already explored or anything similar (e.g. you can't have an "I refuse" style feature or a word count style feature). 

This is open ended, you don't need to use the features you found above, get creative with it! Heck, you can even throw response pairs into your LLM of choice and ask it to come up with differences just like VibeCheck!

In [55]:
# A coherent style feature: direct endings without generic follow-up invitations.
FOLLOW_UP_PHRASES = ['let me know', 'feel free to ask', 'if you have any questions',
                     'if you need further', 'if you need more', 'anything else']
def direct_ending_diff(row):
    def present(conv):
        text = '\n\n'.join(m['content'].lower() for m in conv if m['role'] == 'assistant')
        return int(any(phrase in text for phrase in FOLLOW_UP_PHRASES))
    return present(row['conversation_b']) - present(row['conversation_a'])
selected_battles_no_ties = selected_battles_no_ties.copy()
selected_battles_no_ties['style_direct_ending'] = selected_battles_no_ties.apply(direct_ending_diff, axis=1)
style_feature_cols = ['style_bold_count','style_header_count','style_list_count',
                      'style_sum_assistant_tokens','refusal_count','style_direct_ending']
combined_results_df_new = get_sc_category_results(selected_battles_no_ties, selected_models,
    category_name='Overall w/ Style Control', n_bootstrap=25, style_features=style_feature_cols)
fig = plot_rank_heatmap(pd.concat([results_df, combined_results_df_new]),
    title='With and Without Style Control (New Features)', selected_models=selected_models)
fig.show()
plot_style_features(combined_results_df_new, selected_models).show()
coefs = combined_results_df_new.set_index('Model')['Average Score']
print('Custom coefficient:', coefs['style_direct_ending'], 'Header coefficient:', coefs['style_header_count'])
print('Magnitude criterion:', coefs['style_direct_ending'] > coefs['style_header_count'])
rank_changes = results_df[['Model','Rank']].merge(
    combined_results_df_new.loc[combined_results_df_new['Model'].isin(selected_models), ['Model','Rank']],
    on='Model', suffixes=('_baseline','_controlled'))
rank_changes['improvement'] = rank_changes['Rank_baseline']-rank_changes['Rank_controlled']
display(rank_changes.sort_values('improvement', ascending=False))

Custom coefficient: 0.23759769997879995 Header coefficient: 0.11562184315849042
Magnitude criterion: True


,Model,Rank_baseline,Rank_controlled,improvement
9,claude-3-opus-20240229,9,5,4
17,deepseek-coder-v2,15,12,3
5,claude-3-5-sonnet-20240620,5,2,3
16,claude-3-haiku-20240307,15,13,2
15,gemma-2-9b-it,15,13,2
8,gemini-1.5-pro-api-0514,7,5,2
10,gemini-1.5-flash-api-0514,11,10,1
14,qwen2-72b-instruct,14,13,1
11,gemma-2-27b-it,11,10,1
19,llama-3-8b-instruct,20,19,1


In [56]:
# Memory cleanup: drop conversation columns (saves ~400-600 MB)
# All features extracted from conversations have been computed
selected_battles_no_ties = selected_battles_no_ties.drop(columns=['conversation_a', 'conversation_b'])
import gc
gc.collect()

3362

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

# **Question 9c: Reflection**

**Answer the following questions**

```otter
9c-1. Why did you decide to use the stylistic feature that you have implemented for Q9b? What ranking changes across models did you see when looking at the new style features you created? Why do you think that is the case?

9c-2. Why might some models overuse or underuse stylistic markers (e.g., exclamation points, apologies, or explicit reasoning phrases), and how could that influence rankings?

9c-3. How does including these stylistic features help control for length or formatting effects when building leaderboards?
```

9c-1. I use direct endings without generic follow-up invitations: the feature is +1 when only B includes a phrase such as "let me know", -1 when only A does, and 0 otherwise. It is the reverse orientation of the current notebook's follow-up feature, so a positive coefficient represents preference for a direct ending, conditional on the other covariates. The direct-ending coefficient is approximately 0.238 (95% bootstrap interval [0.170, 0.313]), exceeding the header coefficient of 0.116; this is a joint style adjustment, not proof that changing a closing sentence causes the observed rank changes.

Observed confidence-rank changes include claude-3-5-sonnet-20240620: 5 to 2; llama-3.1-70b-instruct: 5 to 8; claude-3-opus-20240229: 9 to 5; deepseek-coder-v2: 15 to 12; llama-3.1-8b-instruct: 15 to 20.

9c-2. System prompts, training data, and preference tuning can encourage models to repeat enthusiasm, apologies, or reasoning cues. Those habits may influence user votes independently of correctness and may work differently across tasks or audiences.

9c-3. Including length and formatting differences as covariates separates their observed associations from the model-identity coefficients. The resulting ranking compares models at equal measured style values, but unmeasured content differences and correlated features can still confound the interpretation.

<!-- END QUESTION -->

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [57]:
## Use this cell if you are running the notebook in Google Colab to install the necessary dependencies, this may take a few minutes
if IS_COLAB:
    !apt-get install -y texlive texlive-xetex pandoc

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)